In [18]:
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

# Importamos los modelos
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

# Importamos tus utilidades
from data_utils import load_and_clean_compas, obtener_preprocesador, evaluar_rendimiento

#### **4.1 - Preprocesamiento**

In [19]:
X, y = load_and_clean_compas('./dataset/compas-scores-two-years.csv')

stratify_cols = y.astype(str) + X['race'].astype(str)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.30,        
    random_state=42, 
    stratify=stratify_cols
)

preprocessor = obtener_preprocesador()

#### **4.2 - Modelo baseline**

##### Construcción del Pipeline

In [20]:
def optimizar_modelo(nombre_modelo, estimador, param_grid, preprocessor, X_train, y_train):
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', estimador)
    ])
    
    scoring_metrics = {
        'AUC': 'roc_auc', 'Accuracy': 'accuracy',
        'F1': 'f1', 'Precision': 'precision', 'Recall': 'recall'
    }
    
    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
        scoring=scoring_metrics,
        refit='AUC',
        n_jobs=-1,
        verbose=1
    )
    
    print(f"\nIniciando optimización para: {nombre_modelo}")
    grid_search.fit(X_train, y_train)
    
    print(f"\nMejor {nombre_modelo}")
    print(f"AUC-ROC Máximo: {grid_search.best_score_:.4f}")
    
    resultados = grid_search.cv_results_
    df_resultados = pd.DataFrame(resultados['params'])
    df_resultados.columns = [col.replace('classifier__', '') for col in df_resultados.columns]
    
    df_resultados['AUC-ROC'] = resultados['mean_test_AUC']
    df_resultados['Accuracy'] = resultados['mean_test_Accuracy']
    df_resultados['F1-Score'] = resultados['mean_test_F1']
    df_resultados['Precision'] = resultados['mean_test_Precision']
    df_resultados['Recall'] = resultados['mean_test_Recall']
    
    df_top10 = df_resultados.sort_values(by='AUC-ROC', ascending=False).head(10).reset_index(drop=True)
    
    idx_ganador = grid_search.best_index_
    resumen_ganador = {
        'Modelo': nombre_modelo,
        'AUC-ROC': resultados['mean_test_AUC'][idx_ganador],
        'Accuracy': resultados['mean_test_Accuracy'][idx_ganador],
        'F1-Score': resultados['mean_test_F1'][idx_ganador],
        'Precision': resultados['mean_test_Precision'][idx_ganador],
        'Recall': resultados['mean_test_Recall'][idx_ganador],
        'Hiperparámetros': str(grid_search.best_params_).replace('classifier__', '')
    }
    
    return grid_search.best_estimator_, df_top10, resumen_ganador

### Regresión Logística

In [21]:
param_grid_lr = {
    'classifier__C': [0.001, 0.01, 0.1, 1, 10, 100], # Inversa de la regularización
    'classifier__penalty': ['l2'],                   # Penalización Ridge
    'classifier__class_weight': [None, 'balanced']   # Balanceo de clases automático
}

mejor_modelo_lr, df_top10_lr, resumen_lr = optimizar_modelo(
    nombre_modelo="Regresión Logística",
    estimador=LogisticRegression(random_state=42, max_iter=1000),
    param_grid=param_grid_lr,
    preprocessor=preprocessor,
    X_train=X_train,
    y_train=y_train
)

# Mostrar resultados
print("\nTOP 10")
display(df_top10_lr)


Iniciando optimización para: Regresión Logística
Fitting 5 folds for each of 12 candidates, totalling 60 fits

Mejor Regresión Logística
AUC-ROC Máximo: 0.7305

TOP 10


,C,class_weight,penalty,AUC-ROC,Accuracy,F1-Score,Precision,Recall
0,0.010,balanced,l2,0.730454,0.678704,0.642450,0.651383,0.634298
1,0.010,None,l2,0.730313,0.681250,0.589697,0.712192,0.503575
2,0.001,None,l2,0.730163,0.659259,0.499543,0.753255,0.373861
3,0.001,balanced,l2,0.730117,0.681713,0.630582,0.668700,0.597167
4,0.100,balanced,l2,0.729458,0.674306,0.641664,0.642415,0.641429
5,0.100,None,l2,0.729329,0.681250,0.602972,0.696212,0.532060
6,1.000,balanced,l2,0.728617,0.672685,0.639422,0.640966,0.638378
7,1.000,None,l2,0.728432,0.681250,0.607239,0.691472,0.541728
8,10.000,balanced,l2,0.728243,0.671991,0.638490,0.640189,0.637366
9,100.000,balanced,l2,0.728192,0.672222,0.639012,0.640183,0.638382


### Árbol de Decisión

In [22]:
param_grid_dt = {
    'classifier__criterion': ['gini', 'entropy'],
    'classifier__max_depth': [2, 3, 4, 5, 7, None],
    'classifier__min_samples_split': [2, 10, 20, 50],
    'classifier__min_samples_leaf': [1, 5, 10, 20],
    'classifier__class_weight': [None, 'balanced']
}

mejor_modelo_dt, df_top10_dt, resumen_dt = optimizar_modelo(
    nombre_modelo="Árbol de Decisión",
    estimador=DecisionTreeClassifier(random_state=42),
    param_grid=param_grid_dt,
    preprocessor=preprocessor,
    X_train=X_train,
    y_train=y_train
)

print("\nTOP 10")
display(df_top10_dt)


Iniciando optimización para: Árbol de Decisión
Fitting 5 folds for each of 384 candidates, totalling 1920 fits

Mejor Árbol de Decisión
AUC-ROC Máximo: 0.7271

TOP 10


,class_weight,criterion,max_depth,min_samples_leaf,min_samples_split,AUC-ROC,Accuracy,F1-Score,Precision,Recall
0,balanced,gini,5.0,10,2,0.727075,0.687037,0.649623,0.663152,0.637350
1,balanced,gini,5.0,10,20,0.727075,0.687037,0.649623,0.663152,0.637350
2,balanced,gini,5.0,10,10,0.727075,0.687037,0.649623,0.663152,0.637350
3,balanced,gini,4.0,20,2,0.726775,0.686574,0.645224,0.666739,0.626645
4,balanced,gini,4.0,20,50,0.726775,0.686574,0.645224,0.666739,0.626645
5,balanced,gini,4.0,20,20,0.726775,0.686574,0.645224,0.666739,0.626645
6,balanced,gini,4.0,20,10,0.726775,0.686574,0.645224,0.666739,0.626645
7,balanced,gini,4.0,10,50,0.726390,0.686574,0.645224,0.666739,0.626645
8,balanced,gini,4.0,10,10,0.726150,0.686574,0.645224,0.666739,0.626645
9,balanced,gini,4.0,10,2,0.726150,0.686574,0.645224,0.666739,0.626645


### Random Forest

In [23]:
param_grid_rf = {
    'classifier__n_estimators': [50, 100, 200, 300],
    'classifier__max_depth': [2, 3, 5, 7, 10],
    'classifier__max_features': ['sqrt', 'log2', 0.5], # 0.5 significa que usa el 50% de las variables
    'classifier__class_weight': [None, 'balanced', 'balanced_subsample']
}

mejor_modelo_rf, df_top10_rf, resumen_rf = optimizar_modelo(
    nombre_modelo="Random Forest",
    estimador=RandomForestClassifier(random_state=42),
    param_grid=param_grid_rf,
    preprocessor=preprocessor,
    X_train=X_train,
    y_train=y_train
)

print("\nTOP 10")
display(df_top10_rf)


Iniciando optimización para: Random Forest
Fitting 5 folds for each of 180 candidates, totalling 900 fits

Mejor Random Forest
AUC-ROC Máximo: 0.7370

TOP 10


,class_weight,max_depth,max_features,n_estimators,AUC-ROC,Accuracy,F1-Score,Precision,Recall
0,balanced,5,0.5,300,0.736975,0.692130,0.652862,0.670973,0.636318
1,balanced_subsample,5,0.5,300,0.736937,0.691435,0.652374,0.669847,0.636320
2,None,5,0.5,300,0.736778,0.693287,0.638896,0.688565,0.596657
3,balanced_subsample,5,0.5,200,0.736731,0.692824,0.653556,0.671714,0.636829
4,balanced,5,0.5,200,0.736555,0.692361,0.653401,0.670897,0.637336
5,balanced_subsample,5,0.5,100,0.736241,0.692824,0.653417,0.671839,0.636321
6,None,5,0.5,200,0.736199,0.694213,0.639811,0.689798,0.597166
7,None,5,0.5,100,0.736176,0.693056,0.638808,0.688109,0.596659
8,balanced,5,0.5,100,0.736101,0.692361,0.653228,0.670926,0.636829
9,None,5,0.5,50,0.735765,0.692361,0.639685,0.685750,0.600213


### SVM

In [24]:
param_grid_svm = {
    'classifier__C': [0.1, 1, 10],
    'classifier__kernel': ['linear', 'rbf'],
    'classifier__class_weight': [None, 'balanced']
}

mejor_modelo_svm, df_top10_svm, resumen_svm = optimizar_modelo(
    nombre_modelo="SVM",
    estimador=SVC(probability=True, random_state=42),
    param_grid=param_grid_svm,
    preprocessor=preprocessor,
    X_train=X_train,
    y_train=y_train
)

print("\nTOP 10")
display(df_top10_svm)


Iniciando optimización para: SVM
Fitting 5 folds for each of 12 candidates, totalling 60 fits

Mejor SVM
AUC-ROC Máximo: 0.7315

TOP 10


,C,class_weight,kernel,AUC-ROC,Accuracy,F1-Score,Precision,Recall
0,0.1,balanced,rbf,0.731545,0.684491,0.631465,0.674647,0.594105
1,0.1,None,linear,0.730482,0.660648,0.523504,0.724560,0.409979
2,10.0,None,linear,0.730229,0.662269,0.528414,0.724359,0.416082
3,1.0,balanced,rbf,0.730219,0.684722,0.637702,0.669246,0.609875
4,1.0,None,linear,0.730174,0.662500,0.528896,0.724604,0.416591
5,0.1,balanced,linear,0.729060,0.674537,0.620892,0.661609,0.585460
6,0.1,None,rbf,0.728174,0.671065,0.567946,0.706611,0.475078
7,1.0,balanced,linear,0.728034,0.674769,0.629717,0.653869,0.607852
8,10.0,balanced,linear,0.727871,0.674306,0.629237,0.653377,0.607343
9,1.0,None,rbf,0.727584,0.682870,0.600479,0.704456,0.523915


### XGBoost

In [25]:
param_grid_xgb = {
    'classifier__n_estimators': [25, 50, 75, 100],  
    'classifier__max_depth': [2, 3, 4],
    'classifier__learning_rate': [0.03, 0.05, 0.07, 0.1],
    'classifier__subsample': [0.2, 0.3, 0.4, 0.5],  
    'classifier__colsample_bytree': [0.5, 0.7, 1.0]
}

mejor_modelo_xgb, df_top10_xgb, resumen_xgb = optimizar_modelo(
    nombre_modelo="XGBoost",
    estimador=XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss'),
    param_grid=param_grid_xgb,
    preprocessor=preprocessor,
    X_train=X_train,
    y_train=y_train
)

print("\nTOP 10")
display(df_top10_xgb)


Iniciando optimización para: XGBoost
Fitting 5 folds for each of 576 candidates, totalling 2880 fits

Mejor XGBoost
AUC-ROC Máximo: 0.7406

TOP 10


,colsample_bytree,learning_rate,max_depth,n_estimators,subsample,AUC-ROC,Accuracy,F1-Score,Precision,Recall
0,1.0,0.05,2,75,0.2,0.740577,0.691204,0.641371,0.680657,0.606824
1,1.0,0.03,2,100,0.2,0.740423,0.693750,0.646443,0.681426,0.615473
2,1.0,0.05,2,100,0.2,0.740228,0.691667,0.644021,0.678787,0.612929
3,1.0,0.05,3,50,0.3,0.740222,0.694213,0.642296,0.687345,0.603267
4,1.0,0.05,3,50,0.2,0.740148,0.694444,0.642653,0.687210,0.603773
5,1.0,0.07,2,75,0.2,0.740045,0.694444,0.646402,0.683011,0.613951
6,1.0,0.03,3,75,0.2,0.740041,0.694676,0.643357,0.687587,0.605297
7,1.0,0.03,3,50,0.2,0.740020,0.692130,0.637221,0.687475,0.594109
8,1.0,0.07,2,75,0.3,0.739967,0.692361,0.643781,0.680897,0.610892
9,1.0,0.03,3,50,0.3,0.739958,0.691435,0.635771,0.687223,0.592078


### Resultados Globales

In [26]:
lista_resumenes = [resumen_lr, resumen_dt, resumen_rf, resumen_svm, resumen_xgb]

df_comparativa_final = pd.DataFrame(lista_resumenes)

df_comparativa_final = df_comparativa_final.sort_values(by='AUC-ROC', ascending=False).reset_index(drop=True)

display(df_comparativa_final.drop(columns=['Hiperparámetros']))

print("\nHiperparámetros Ganadores:")
for i, row in df_comparativa_final.iterrows():
    print(f"{row['Modelo']}: {row['Hiperparámetros']}")

,Modelo,AUC-ROC,Accuracy,F1-Score,Precision,Recall
0,XGBoost,0.740577,0.691204,0.641371,0.680657,0.606824
1,Random Forest,0.736975,0.692130,0.652862,0.670973,0.636318
2,SVM,0.731545,0.684491,0.631465,0.674647,0.594105
3,Regresión Logística,0.730454,0.678704,0.642450,0.651383,0.634298
4,Árbol de Decisión,0.727075,0.687037,0.649623,0.663152,0.637350



Hiperparámetros Ganadores:
XGBoost: {'colsample_bytree': 1.0, 'learning_rate': 0.05, 'max_depth': 2, 'n_estimators': 75, 'subsample': 0.2}
Random Forest: {'class_weight': 'balanced', 'max_depth': 5, 'max_features': 0.5, 'n_estimators': 300}
SVM: {'C': 0.1, 'class_weight': 'balanced', 'kernel': 'rbf'}
Regresión Logística: {'C': 0.01, 'class_weight': 'balanced', 'penalty': 'l2'}
Árbol de Decisión: {'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 5, 'min_samples_leaf': 10, 'min_samples_split': 2}
